# 4. Community Detection & Summarization Pipeline

Notebook ini menjalankan algoritma **Leiden** (via GDS) untuk mendeteksi komunitas di dalam Knowledge Graph, kemudian men-generate **ringkasan (summary) komunitas** menggunakan LLM.

Ringkasan ini sangat penting untuk mendukung fitur **Global Search** (mendapatkan insight tingkat makro) dan **DRIFT Search**.

In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from src.graph.client import GraphClient
from src.config import settings
from src.community.detection import CommunityDetector
from src.community.summarization import CommunitySummarizer
from src.services.providers import get_llm, get_embeddings

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
client = GraphClient(
    uri=settings.neo4j_uri,
    user=settings.neo4j_user,
    password=settings.neo4j_password
)

## Konfigurasi Graph

Node dan relationship dideteksi otomatis dari default **`detection.py`** (machine_labels) dan **`config.py`** (`community_relationship_types`).
Tidak perlu hardcode di notebook — semua sinambung dari satu sumber.

## Step 1: Detect Communities via Leiden
*(Catatan: Membutuhkan plugin Neo4j GDS terinstall)*

In [ ]:
detector = CommunityDetector(client)
success = detector.detect(force=True)

if success:
    print("Community detection successful!")
else:
    print("Community detection failed. Please check if GDS is installed.")

## Step 2: Summarize Communities via LLM
Proses ini akan mengirimkan entitas di tiap komunitas menggunakan Azure OpenAI untuk dirangkum. Waktu eksekusi bergantung pada jumlah komunitas dan limit rate Azure OpenAI.

In [ ]:
llm = get_llm(temperature=0.2)
embedder = get_embeddings()

summarizer = CommunitySummarizer(client, llm, embedder, max_workers=4)
summarizer.summarize_all()
print("Community summarization completed!")

In [ ]:
client.close()